# Scratch — exploration only

> ## ⚠️ NOT PART OF THE PIPELINE
>
> **Nothing in this notebook feeds the report.** No figure here is saved to
> `figures/`, no number here is quoted anywhere, and
> `scripts/make_all_figures.py` does not touch it.
>
> It exists so that exploratory work has somewhere to live that is *obviously*
> separate from the reproducible pipeline. If something explored here turns out
> to matter, the right move is to promote it into an experiment script under
> `experiments/` with a test, not to cite it from here.

---

Below are a few starting points, left deliberately open.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import matplotlib.pyplot as plt

from fibroblock import config as cfg
from fibroblock import fhn, grid, measure, operators, plotting, simulate, solvers, utils

config = cfg.default_config()

## Idea 1 — two gaps instead of one

The model supports only a single gap through `GapParams`. Two gaps in series
would need `grid.diffusion_profile` extended, but the operator and solver would
not change at all — the conservative form handles any `D(x)`.

Question worth asking: do two 0.05 cm gaps block more or less readily than one
0.1 cm gap? The saturation result in `ex06` suggests less readily, since each is
below the saturation length.

In [2]:
# Sketch: build a custom D(x) directly and pass it to the operator.
x = np.linspace(0.0, 2.0, 201)
D = np.full_like(x, 0.001)
D[(x >= 0.90) & (x <= 0.95)] = 0.15 * 0.001
D[(x >= 1.05) & (x <= 1.10)] = 0.15 * 0.001

D_half = grid.half_node_diffusion(D, "harmonic")

fig, ax = plotting.new_figure()
ax.step(x, D / 0.001, where="mid")
plotting.label_axes(ax, "position $x$ (cm)", r"$D(x)/D_0$ (dimensionless)", "Two gaps in series")
plt.show()

C:\Users\dippe\AppData\Local\Temp\ipykernel_2436\2080556031.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Idea 2 — a graded rather than sharp gap edge

Assumption A8 says the gap edge is a sharp step. Real fibrosis has graded
borders. Smoothing the transition over a few space constants should make block
*harder*, because the wave is no longer asked to cross a discontinuity.

In [3]:
# Sketch: a tanh-smoothed profile with the same minimum and the same width.
def smoothed_profile(x, rho, centre, half_width, edge_width):
    """Tanh-smoothed coupling profile; edge_width -> 0 recovers the step."""
    left = np.tanh((x - (centre - half_width)) / edge_width)
    right = np.tanh(((centre + half_width) - x) / edge_width)
    well = 0.5 * (left + right)  # ~1 inside the gap, ~0 outside
    return 0.001 * (1.0 - (1.0 - rho) * np.clip(well, 0.0, 1.0))


fig, ax = plotting.new_figure()
for edge in (0.001, 0.01, 0.03):
    ax.plot(x, smoothed_profile(x, 0.15, 1.0, 0.05, edge) / 0.001, label=f"edge width {edge} cm")
plotting.label_axes(ax, "position $x$ (cm)", r"$D(x)/D_0$ (dimensionless)", "Graded gap edges")
ax.legend(fontsize=8)
plt.show()

C:\Users\dippe\AppData\Local\Temp\ipykernel_2436\2111385666.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Idea 3 — a premature second stimulus

The whole project simulates a single beat, which is why nothing is claimed about
re-entry. A premature stimulus arriving while the tissue is partly refractory
would raise $w$ locally and so lift the effective threshold — the same mechanism
that produces the delay ceiling in `ex07`.

**Caveat worth remembering before spending time on this:** FitzHugh–Nagumo's
action potential is 4–6× shorter than a real ventricular one (see
`docs/validation_log.md` L5), so anything depending on refractoriness would be
qualitative at best.

In [4]:
# Space for exploration.


## Idea 4 — how much does the activation-time definition actually matter?

Both definitions are computed on every run, so this is a two-line comparison
rather than a re-run. Assumption A14 says they differ by a fraction of a
millisecond; worth confirming for the report's own statement.

In [5]:
result = simulate.run_simulation(config.replace(gap=cfg.GapParams(rho=1.0, gap_length_cm=0.0)))

difference = result.activation_time_max_dvdt - result.activation_time_crossing
usable = np.isfinite(difference)
print(f"max |difference| between the two activation definitions: {np.nanmax(np.abs(difference)):.4f} ms")
print(f"mean difference: {np.nanmean(difference[usable]):+.4f} ms")

by_crossing = measure.fit_conduction_velocity(
    result.x, result.activation_time_crossing, config.measurement, config.grid.length_cm
)
by_max_dvdt = measure.fit_conduction_velocity(
    result.x, result.activation_time_max_dvdt, config.measurement, config.grid.length_cm
)
print(f"theta by V=0 crossing : {by_crossing.theta_cm_per_ms:.7f} cm/ms")
print(f"theta by max dV/dt    : {by_max_dvdt.theta_cm_per_ms:.7f} cm/ms")
print(f"relative difference   : "
      f"{100 * abs(by_crossing.theta_cm_per_ms - by_max_dvdt.theta_cm_per_ms) / by_crossing.theta_cm_per_ms:.3f} %")

max |difference| between the two activation definitions: 3.4040 ms
mean difference: +0.3048 ms
theta by V=0 crossing : 0.0255336 cm/ms
theta by max dV/dt    : 0.0255336 cm/ms
relative difference   : 0.000 %


In [6]:
# Scratch space below this line.
